# IndexGamma Pitch EDA: is market intraday momentum (leveraged-ETF and dealer gamma) still there?

**The papers.** Gao, Han, Li & Zhou (*JFE* 2018): the first half-hour return predicts the last half-hour return of the
S&P 500. Baltussen, Da, Lammers & Martens (*JFE* 2021) tie it to gamma hedging by leveraged ETFs and option dealers:
the return so far predicts the last 30 minutes. Both samples end before 2021.

**Design.** SPY and QQQ 5-minute bars (Twelve Data, 2020-02 →; `collect_intraday.py`), official closes from Yahoo.
Signals: previous close → 10:00 (first30) and previous close → 15:30 (rest). Outcome: 15:30 → official close. Trade:
sign of the signal, net 1bp. Also: does the last 30 minutes reverse overnight? 2020 vs 2021–23 vs 2024–.

In [1]:
import pandas as pd, numpy as np, statsmodels.api as sm
px=pd.read_csv("../TOM Pitch/data/etfs.csv",parse_dates=["date"])
cl=px.pivot(index="date",columns="ticker",values="close")
for s in ["SPY","QQQ"]:
    x=pd.read_csv(f"data/intraday/{s}.csv",parse_dates=["datetime"]); x["d"]=x.datetime.dt.normalize(); x["hm"]=x.datetime.dt.strftime("%H:%M")
    c=x.pivot_table(index="d",columns="hm",values="close"); c=c[c.notna().sum(1)>=76]
    o930=x[x.hm=="09:30"].set_index("d").open.reindex(c.index)          # first-bar open = official open (approx.)
    C=cl[s].reindex(c.index); P=cl[s].shift(1).reindex(c.index)
    D=pd.DataFrame({"first30":np.log(c["09:55"]/P)*1e4,"rest":np.log(c["15:25"]/P)*1e4,"last30":np.log(C/c["15:25"])*1e4,
                    "next_on":np.log(o930.shift(-1)/C)*1e4}).dropna()
    for per,m in [("2020",D.index<"2021"),("2021-2023",(D.index>="2021")&(D.index<"2024")),("2024-",D.index>="2024")]:
        d=D[m]; out=[]
        for sig in ["first30","rest"]:
            f=sm.OLS(d.last30,sm.add_constant(d[sig])).fit(cov_type="HC1")
            pnl=np.sign(d[sig])*d.last30-1.0
            out.append(f"{sig} -> last30: b {f.params[sig]:+.3f} (t {f.tvalues[sig]:+.2f}); trade net1bp Sharpe {pnl.mean()/pnl.std()*np.sqrt(252):+.2f}")
        g=sm.OLS(d.next_on,sm.add_constant(d.last30)).fit(cov_type="HC1")
        print(s,per,"n",len(d)," | ".join(out), f"| overnight reversal of last30: b {g.params.last30:+.2f} (t {g.tvalues.last30:+.2f})")

SPY 2020 n 220 first30 -> last30: b +0.084 (t +1.34); trade net1bp Sharpe +0.71 | rest -> last30: b +0.035 (t +0.66); trade net1bp Sharpe +1.44 | overnight reversal of last30: b -1.02 (t -2.73)
SPY 2021-2023 n 740 first30 -> last30: b -0.026 (t -1.36); trade net1bp Sharpe -1.15 | rest -> last30: b +0.011 (t +0.94); trade net1bp Sharpe +0.53 | overnight reversal of last30: b -0.03 (t -0.29)
SPY 2024- n 692 first30 -> last30: b -0.010 (t -0.50); trade net1bp Sharpe -0.51 | rest -> last30: b +0.008 (t +0.33); trade net1bp Sharpe -2.01 | overnight reversal of last30: b -0.10 (t -0.45)


QQQ 2020 n 219 first30 -> last30: b +0.145 (t +1.81); trade net1bp Sharpe +1.11 | rest -> last30: b +0.063 (t +1.09); trade net1bp Sharpe +1.13 | overnight reversal of last30: b -0.82 (t -2.95)
QQQ 2021-2023 n 742 first30 -> last30: b -0.019 (t -1.24); trade net1bp Sharpe -1.77 | rest -> last30: b +0.008 (t +0.86); trade net1bp Sharpe +0.44 | overnight reversal of last30: b -0.07 (t -0.61)
QQQ 2024- n 693 first30 -> last30: b -0.002 (t -0.15); trade net1bp Sharpe -0.41 | rest -> last30: b -0.002 (t -0.16); trade net1bp Sharpe -1.76 | overnight reversal of last30: b -0.12 (t -0.64)


**Interpretation.** The effect is visible in 2020 (QQQ first30 t 1.8; trades Sharpe 0.7–1.4, along with a strong
overnight reversal of the last 30 minutes, t ≈ −2.8). **From 2021 it is gone**: slopes ≈ 0 (|t| ≤ 1.4) for both
signals and both ETFs, the trade Sharpe is negative in 2024–, and the overnight reversal disappears. That fits the
single-stock leveraged-ETF result: the rebalancing flow is anticipated and absorbed.

**Verdict: dead after publication.**